In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('cleaned_data.csv')

X = df.drop(columns=['price'])
y_reg = df['price']
y_clf = (y_reg > y_reg.median()).astype(int)

cut_order = {'Fair': 0, 'Good':1, 'Very Good':2, 'Premium': 3, 'Ideal':4}
color_order = {'J':0, 'I':1, 'H':2, 'G':3, 'F':4, 'E':5, 'D':6}
clarity_order = {'I1':0, 'SI2':1, 'SI1':2, 'VS2':3, 'VS1':4, 'VVS2':5, 'VVS1':6, 'IF':7}

X['cut'] = X['cut'].map(cut_order)
X['color'] = X['color'].map(color_order)
X['clarity'] = X['clarity'].map(clarity_order)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Setup complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

Setup complete.
X_train_scaled shape: (43035, 9)
X_test_scaled shape: (10759, 9)


In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled))

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Train accuracy: 0.9999302892994074
Test accuracy: 0.9736964401896087


In [5]:
# controlled decision tree
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_controlled = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_controlled = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print("Controlled Train accuracy:", train_acc_controlled)
print("Controlled Test accuracy:", test_acc_controlled)

Controlled Train accuracy: 0.971953061461601
Controlled Test accuracy: 0.9710010223998513


In [6]:
# Gini vs Entropy comparison
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print("Gini test accuracy:", test_acc_gini)
print("Entropy test accuracy:", test_acc_entropy)

Gini test accuracy: 0.9710010223998513
Entropy test accuracy: 0.9602193512408216


In [7]:
# Random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf.predict(X_test_scaled))
rf_auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1])

print("RF Train accuracy:", rf_train_acc)
print("RF Test accuracy:", rf_test_acc)
print("RF ROC-AUC:", rf_auc)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)

print(feature_importance.head(5))

RF Train accuracy: 0.9894272104101313
RF Test accuracy: 0.9796449484152803
RF ROC-AUC: 0.9984678187687189
   feature  importance
7        y    0.384645
6        x    0.211499
0    carat    0.198977
8        z    0.120923
3  clarity    0.037944


In [8]:
# t-4a. gradient boosting.
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_clf_test, gb.predict(X_test_scaled))
gb_auc = roc_auc_score(y_clf_test, gb.predict_proba(X_test_scaled)[:, 1])

print("GB Train accuracy:", gb_train_acc)
print("GB Test accuracy:", gb_test_acc)
print("GB ROC-AUC:", gb_auc)

GB Train accuracy: 0.9807133728360636
GB Test accuracy: 0.9785296031229668
GB ROC-AUC: 0.9984221000676277


In [9]:
# task - 4b. featture ablation task.
lowest_5 = feature_importance.tail(5)
print("5 lowest-importance features:")
print(lowest_5)

features_to_drop = lowest_5['feature'].tolist()
print("\nDropping:", features_to_drop)

X_train_reduced = pd.DataFrame(X_train_scaled, columns=X.columns).drop(columns=features_to_drop)
X_test_reduced = pd.DataFrame(X_test_scaled, columns=X.columns).drop(columns=features_to_drop)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1])
auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print("Full model AUC:", auc_full)
print("Reduced model AUC (5 lowest-importance features removed):", auc_reduced)

5 lowest-importance features:
   feature  importance
3  clarity    0.037944
2    color    0.030585
4    depth    0.006042
1      cut    0.004930
5    table    0.004456

Dropping: ['clarity', 'color', 'depth', 'cut', 'table']
Full model AUC: 0.9984678187687189
Reduced model AUC (5 lowest-importance features removed): 0.9917934585972755


In [11]:
# task - 5 cross-validated comparison
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree (controlled)': DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring='roc_auc')
    cv_results.append({'Model': name, 'Mean AUC': scores.mean(), 'Std AUC': scores.std()})
    print(name, "scores:", scores)

cv_results_df = pd.DataFrame(cv_results)
print(cv_results_df)

Logistic Regression scores: [0.9976526  0.99762452 0.99724769 0.99759569 0.99758637]
Decision Tree (controlled) scores: [0.99549234 0.99522228 0.99533733 0.99489013 0.99681389]
Random Forest scores: [0.99850421 0.99861275 0.99832306 0.99818073 0.99885376]
Gradient Boosting scores: [0.99846779 0.99853661 0.99834725 0.99824158 0.99876118]
                        Model  Mean AUC   Std AUC
0         Logistic Regression  0.997541  0.000149
1  Decision Tree (controlled)  0.995551  0.000662
2               Random Forest  0.998495  0.000233
3           Gradient Boosting  0.998471  0.000177


In [12]:
# ask-6 hyperparameter tuning with GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train, y_clf_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV AUC score:", grid_search.best_score_)

Best parameters: {'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 200}
Best CV AUC score: 0.9985533192317952


In [13]:
n_combinations = 3 * 3 * 2
total_fits = n_combinations * 5
print(f"Total hyperparameter combinations: {n_combinations}")
print(f"Total model fits: {total_fits}")

Total hyperparameter combinations: 18
Total model fits: 90


In [14]:
best_pipeline = grid_search.best_estimator_

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_results = []

for f in fractions:
    n_rows = int(f * len(X_train))
    X_subset = X_train.iloc[:n_rows]
    y_subset = y_clf_train.iloc[:n_rows]

    best_pipeline.fit(X_subset, y_subset)

    train_proba = best_pipeline.predict_proba(X_subset)[:, 1]
    train_auc = roc_auc_score(y_subset, train_proba)

    test_proba = best_pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_clf_test, test_proba)

    learning_curve_results.append({
        'Training fraction': f,
        'Training AUC': train_auc,
        'Test AUC': test_auc
    })
    print(f"Fraction {f}: Train AUC = {train_auc:.5f}, Test AUC = {test_auc:.5f}")

learning_curve_df = pd.DataFrame(learning_curve_results)
print(learning_curve_df)

Fraction 0.2: Train AUC = 0.99951, Test AUC = 0.99802
Fraction 0.4: Train AUC = 0.99958, Test AUC = 0.99834
Fraction 0.6: Train AUC = 0.99958, Test AUC = 0.99843
Fraction 0.8: Train AUC = 0.99959, Test AUC = 0.99850
Fraction 1.0: Train AUC = 0.99960, Test AUC = 0.99853
   Training fraction  Training AUC  Test AUC
0                0.2      0.999514  0.998019
1                0.4      0.999578  0.998340
2                0.6      0.999584  0.998429
3                0.8      0.999587  0.998504
4                1.0      0.999596  0.998526


In [15]:
# Serialize the best model
import joblib
joblib.dump(best_pipeline, 'best_model.pkl')
print("Model saved as best_model.pkl")

Model saved as best_model.pkl


In [16]:
loaded_model = joblib.load('best_model.pkl')
print("Model loaded successfully.")

new_diamond_1 = pd.DataFrame({
    'carat': [1.5],
    'cut': [4],
    'color': [6],
    'clarity': [5],
    'depth': [61.5],
    'table': [57.0],
    'x': [7.3],
    'y': [7.3],
    'z': [4.5]
})

new_diamond_2 = pd.DataFrame({
    'carat': [0.3],
    'cut': [1],
    'color': [1],
    'clarity': [1],
    'depth': [62.0],
    'table': [58.0],
    'x': [4.3],
    'y': [4.3],
    'z': [2.7]
})

test_rows = pd.concat([new_diamond_1, new_diamond_2], ignore_index=True)

predictions = loaded_model.predict(test_rows)
probabilities = loaded_model.predict_proba(test_rows)[:, 1]

print("Predictions (0=not expensive, 1=expensive):", predictions)
print("Predicted probabilities of being expensive:", probabilities)

Model loaded successfully.
Predictions (0=not expensive, 1=expensive): [1 0]
Predicted probabilities of being expensive: [1. 0.]


In [17]:
summary_table = pd.DataFrame({
    'Model': [
        'Logistic Regression (Part 2)',
        'Decision Tree (controlled)',
        'Random Forest (default)',
        'Gradient Boosting',
        'Random Forest (GridSearchCV tuned)'
    ],
    '5-fold CV Mean AUC': [0.99754, 0.99555, 0.99850, 0.99847, grid_search.best_score_],
    '5-fold CV Std AUC': [0.000149, 0.000662, 0.000233, 0.000177, None],
    'Test-set AUC': [0.9974, None, 0.99847, 0.99842, learning_curve_df['Test AUC'].iloc[-1]]
})
print(summary_table)

                                Model  5-fold CV Mean AUC  5-fold CV Std AUC  \
0        Logistic Regression (Part 2)            0.997540           0.000149   
1          Decision Tree (controlled)            0.995550           0.000662   
2             Random Forest (default)            0.998500           0.000233   
3                   Gradient Boosting            0.998470           0.000177   
4  Random Forest (GridSearchCV tuned)            0.998553                NaN   

   Test-set AUC  
0      0.997400  
1           NaN  
2      0.998470  
3      0.998420  
4      0.998526  


In [18]:
import os
size_bytes = os.path.getsize('best_model.pkl')
size_mb = size_bytes / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

File size: 13.55 MB
